In [1]:
# -*- coding: utf-8 -*-
"""
=============================================================================
S_TailBehavior_Daily — 尾部行为因子 (Ts 高阶矩维度)
=============================================================================
核心思路：
  全场因子都在测"分布中心"（均值/中位数/标准差）
  尾部行为测"分布的尾巴有多肥"——偏度、极端bar频率、日内最大回撤
  → 和 Amp/Volatility 等均值层面因子零共线

数据源：bigalpha_2026_stock_bar1m
信号逻辑：
  1. 日内分钟收益偏度 → 测"分布不对称性"
  2. 极端 bar 频率（超过 ±2σ 的 bar 占比）→ 测"尾部厚度"
  3. 日内最大回撤深度 → 测"最坏情况的深度"
  4. 三者截面 rank + 中位数融合 → factor
=============================================================================
"""

import dai
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')


def main(datasources, start_date, end_date):
    """BigQuant 标准入口：datasources 由平台注入，start_date/end_date 为回测区间"""
    bar1m = datasources["bar1m"]
    query_start = (pd.to_datetime(start_date) - pd.Timedelta(days=90)).strftime('%Y-%m-%d %H:%M:%S')

    # ------------------------------ 1. SQL：分三步走，避免窗口函数嵌套 ------------------------------
    sql = f"""
    WITH min_returns AS (
        SELECT
            date,
            instrument,
            close,
            high,
            low,
            date_trunc('day', date) AS day_date,
            (close - LAG(close) OVER (
                PARTITION BY instrument, date_trunc('day', date) ORDER BY date
            )) / NULLIF(LAG(close) OVER (
                PARTITION BY instrument, date_trunc('day', date) ORDER BY date
            ), 0) AS minute_ret
        FROM {bar1m}
        WHERE close > 0
    ),
    -- 第二步：先算每根bar的均值/标准差（窗口函数，在聚合之前）
    with_stats AS (
        SELECT
            *,
            AVG(minute_ret) OVER (PARTITION BY instrument, day_date) AS day_mean_ret,
            STDDEV(minute_ret) OVER (PARTITION BY instrument, day_date) AS day_std_ret
        FROM min_returns
        WHERE minute_ret IS NOT NULL
    ),
    -- 第三步：用预计算的值做聚合（不再嵌套窗口函数）
    daily_stats AS (
        SELECT
            day_date AS date,
            instrument,
            AVG(minute_ret) AS mean_ret,
            STDDEV(minute_ret) AS std_ret,
            COUNT(minute_ret) AS n_valid,
            COUNT(CASE WHEN minute_ret > day_mean_ret + 2 * day_std_ret THEN 1 END) AS extreme_high_bars,
            COUNT(CASE WHEN minute_ret < day_mean_ret - 2 * day_std_ret THEN 1 END) AS extreme_low_bars,
            MAX(high) AS day_high,
            MIN(low) AS day_low
        FROM with_stats
        GROUP BY day_date, instrument
        HAVING COUNT(minute_ret) >= 60
    )
    SELECT
        date,
        instrument,
        mean_ret,
        std_ret,
        n_valid,
        extreme_high_bars,
        extreme_low_bars,
        day_high,
        day_low
    FROM daily_stats
    WHERE std_ret > 0
    """

    df = dai.query(sql, filters={'date': [query_start, end_date]}).df()
    df['date'] = pd.to_datetime(df['date'])

    # ------------------------------ 2. Python：计算尾部统计量 ------------------------------
    for col in ['mean_ret', 'std_ret', 'n_valid', 'extreme_high_bars',
                'extreme_low_bars', 'day_high', 'day_low']:
        df[col] = df[col].astype(float)

    # A. 偏度近似：mean / (std / sqrt(n))
    df['skew_sign'] = df['mean_ret'] / (
        df['std_ret'].clip(lower=1e-8) / np.sqrt(df['n_valid'].clip(lower=1))
    )

    # B. 极端 bar 频率（双侧）
    df['extreme_freq'] = (
        df['extreme_high_bars'] + df['extreme_low_bars']
    ) / df['n_valid'].clip(lower=1)

    # C. 日内最大回撤深度：(high - low) / high
    df['drawdown_depth'] = (df['day_high'] - df['day_low']) / df['day_high'].clip(lower=1e-8)

    # ------------------------------ 3. 多日平滑（10日滚动中位数） ------------------------------
    df = df.sort_values(['instrument', 'date'])

    def smooth_tail(group):
        group = group.copy()
        group['skew_sign_smooth'] = group['skew_sign'].rolling(10, min_periods=3).median()
        group['extreme_freq_smooth'] = group['extreme_freq'].rolling(10, min_periods=3).median()
        group['drawdown_smooth'] = group['drawdown_depth'].rolling(10, min_periods=3).median()
        return group

    df = df.groupby('instrument', group_keys=False).apply(smooth_tail)

    # ------------------------------ 4. 每日截面 rank ------------------------------
    def daily_rank(group):
        for col in ['skew_sign_smooth', 'extreme_freq_smooth', 'drawdown_smooth']:
            group[f'{col}_rank'] = group[col].rank(pct=True)
        return group

    df = df.groupby('date', group_keys=False).apply(daily_rank)

    # ------------------------------ 5. 构建因子 ------------------------------
    df['raw_factor'] = df[[
        'skew_sign_smooth_rank',
        'extreme_freq_smooth_rank',
        'drawdown_smooth_rank'
    ]].median(axis=1)

    # 高尾部风险 → 预期未来收益低 → 取负
    df['factor'] = 1 - df['raw_factor']

    # ------------------------------ 6. 对齐中证1000成分股 + 输出 ------------------------------
    stk_pool = dai.query(
        "SELECT date, instrument FROM bigalpha_2026_instruments",
        filters={'date': [start_date, end_date]}
    ).df()
    stk_pool['date'] = pd.to_datetime(stk_pool['date'])

    result = pd.merge(stk_pool, df[['date', 'instrument', 'factor']],
                      on=['date', 'instrument'], how='left')
    result['factor'] = result['factor'].fillna(0.5).replace([np.inf, -np.inf], 0.5).clip(0, 1)

    result = result[
        (result['date'] >= pd.to_datetime(start_date)) &
        (result['date'] <= pd.to_datetime(end_date))
    ]

    return result[['date', 'instrument', 'factor']]